In [ ]:
import gzip
from collections.abc import Callable, Iterable
from functools import partial
from typing import NamedTuple

import numpy as np
import scipy.sparse
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR
from plot_tools import (
    draw_multi_pmf_cdf_fig,
    draw_pmf_cdf_fig,
    print_distribution_table,
)
from random_variable import FiniteDist


构造状态转移矩阵的 python 代码

In [ ]:
class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    UP六星水位: int
    已获取UP六星干员数量: int
    已获取UP五星干员数量: int


def 获取状态(
    *,
    六星水位: int,
    五星水位: int,
    UP六星水位: int,
    已获取UP六星干员数量: int,
    已获取UP五星干员数量: int,
) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        UP六星水位=UP六星水位,
        已获取UP六星干员数量=min(已获取UP六星干员数量, 已获取UP六星干员数量上限),
        已获取UP五星干员数量=min(已获取UP五星干员数量, 已获取UP五星干员数量上限),
    )


def 状态转移(
    起始状态: 状态类,
    *,
    是第10抽: bool,
    前50抽但非第10抽: bool,
    第51抽及以后: bool,
) -> list[tuple[状态类, float]]:
    if int(是第10抽) + int(前50抽但非第10抽) + int(第51抽及以后) != 1:
        raise ValueError("必须且只能指定一个抽数阶段")

    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位
    起始UP六星水位 = 起始状态.UP六星水位
    起始已获取UP六星干员数量 = 起始状态.已获取UP六星干员数量
    起始已获取UP五星干员数量 = 起始状态.已获取UP五星干员数量

    if 起始UP六星水位 == 119:
        # 连续 119 抽未抽到 UP 6★ 干员，下一抽必定抽到 UP 6★ 干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=0,
                    五星水位=0,
                    UP六星水位=0,
                    已获取UP六星干员数量=起始已获取UP六星干员数量 + 1,
                    已获取UP五星干员数量=起始已获取UP五星干员数量,
                ),
                1,
            )
        )

    else:
        # 计算抽到不同星级干员的概率
        六星概率 = COND_PROB_6_STAR[起始六星水位]
        if 是第10抽 and 起始五星水位 == 9:
            五星概率 = 1 - 六星概率
        else:
            五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
        四星或三星概率 = 1 - 六星概率 - 五星概率

        if 前50抽但非第10抽 and 起始已获取UP五星干员数量 == 0:
            UP五星概率 = 五星概率
            其他五星概率 = 0
        else:
            UP五星概率 = 五星概率 / 2
            其他五星概率 = 五星概率 / 2

        # 抽到UP6星干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=0,
                    五星水位=0,
                    UP六星水位=0,
                    已获取UP六星干员数量=起始已获取UP六星干员数量 + 1,
                    已获取UP五星干员数量=起始已获取UP五星干员数量,
                ),
                六星概率 / 2,
            )
        )

        # 抽到其他6星干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=0,
                    五星水位=0,
                    UP六星水位=起始UP六星水位 + 1,
                    已获取UP六星干员数量=起始已获取UP六星干员数量,
                    已获取UP五星干员数量=起始已获取UP五星干员数量,
                ),
                六星概率 / 2,
            )
        )

        # 抽到UP五星干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=起始六星水位 + 1,
                    五星水位=0,
                    UP六星水位=起始UP六星水位 + 1,
                    已获取UP六星干员数量=起始已获取UP六星干员数量,
                    已获取UP五星干员数量=起始已获取UP五星干员数量 + 1,
                ),
                UP五星概率,
            )
        )

        # 抽到其他五星干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=起始六星水位 + 1,
                    五星水位=0,
                    UP六星水位=起始UP六星水位 + 1,
                    已获取UP六星干员数量=起始已获取UP六星干员数量,
                    已获取UP五星干员数量=起始已获取UP五星干员数量,
                ),
                其他五星概率,
            )
        )

        # 抽到四星及以下干员
        转移概率列表.append(
            (
                获取状态(
                    六星水位=起始六星水位 + 1,
                    五星水位=起始五星水位 + 1,
                    UP六星水位=起始UP六星水位 + 1,
                    已获取UP六星干员数量=起始已获取UP六星干员数量,
                    已获取UP五星干员数量=起始已获取UP五星干员数量,
                ),
                四星或三星概率,
            )
        )

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]
    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(
    状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]],
) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


已获取UP六星干员数量上限 = 6
已获取UP五星干员数量上限 = 6


状态列表: list[状态类] = []
状态列表.extend(
    状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        UP六星水位=UP六星水位,
        已获取UP六星干员数量=已获取UP六星干员数量,
        已获取UP五星干员数量=已获取UP五星干员数量,
    )
    for 六星水位 in range(0, 99, 1)
    for 五星水位 in range(0, 40, 1)
    for UP六星水位 in range(0, 120, 1)
    for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限 + 1, 1)
    for 已获取UP五星干员数量 in range(0, 已获取UP五星干员数量上限 + 1, 1)
)
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}


In [ ]:
状态转移矩阵_第10抽 = 构造状态转移矩阵(
    partial(状态转移, 是第10抽=True, 前50抽但非第10抽=False, 第51抽及以后=False)
)
状态转移矩阵_前50抽但非第10抽 = 构造状态转移矩阵(
    partial(状态转移, 是第10抽=False, 前50抽但非第10抽=True, 第51抽及以后=False)
)
状态转移矩阵_第51抽及以后 = 构造状态转移矩阵(
    partial(状态转移, 是第10抽=False, 前50抽但非第10抽=False, 第51抽及以后=True)
)

scipy.sparse.save_npz(
    "联动寻访怪猎二期中间结果/状态转移矩阵_第10抽.npz", 状态转移矩阵_第10抽
)
scipy.sparse.save_npz(
    "联动寻访怪猎二期中间结果/状态转移矩阵_前50抽但非第10抽.npz",
    状态转移矩阵_前50抽但非第10抽,
)
scipy.sparse.save_npz(
    "联动寻访怪猎二期中间结果/状态转移矩阵_第51抽及以后.npz", 状态转移矩阵_第51抽及以后
)

计算历史分布的 python 代码

In [ ]:
中间结果目录 = "联动寻访怪猎二期中间结果"
npz_files = [
    f"{中间结果目录}/状态转移矩阵_第10抽.npz",
    f"{中间结果目录}/状态转移矩阵_前50抽但非第10抽.npz",
    f"{中间结果目录}/状态转移矩阵_第51抽及以后.npz",
]

# 从 .bin 文件读取 → csr_array → 保存为 .npz
def load_csr_from_bin(prefix: str) -> scipy.sparse.csr_array:
    data = np.fromfile(f"{prefix}.data.bin", dtype=np.float64)
    print(f"  已加载 {prefix}.data.bin ({len(data)} 个元素)")
    row_ind = np.fromfile(f"{prefix}.row_ind.bin", dtype=np.uint32)
    print(f"  已加载 {prefix}.row_ind.bin ({len(row_ind)} 个元素)")
    col_ind = np.fromfile(f"{prefix}.col_ind.bin", dtype=np.uint32)
    print(f"  已加载 {prefix}.col_ind.bin ({len(col_ind)} 个元素)")
    size = max(np.max(row_ind) + 1, np.max(col_ind) + 1)
    array = scipy.sparse.csr_array((data, (row_ind, col_ind)), shape=(size, size))
    print(f"  已构造 csr_array，形状 {array.shape}，非零元数量 {array.nnz}")
    return array

# 从 .bin.gz 文件读取 → csr_array → 保存为 .npz
def load_csr_from_bin_gz(prefix: str) -> scipy.sparse.csr_array:
    with gzip.open(f"{prefix}.data.bin.gz", "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.float64)
    print(f"  已加载 {prefix}.data.bin.gz ({len(data)} 个元素)")
    with gzip.open(f"{prefix}.row_ind.bin.gz", "rb") as f:
        row_ind = np.frombuffer(f.read(), dtype=np.uint32)
    print(f"  已加载 {prefix}.row_ind.bin.gz ({len(row_ind)} 个元素)")
    with gzip.open(f"{prefix}.col_ind.bin.gz", "rb") as f:
        col_ind = np.frombuffer(f.read(), dtype=np.uint32)
    print(f"  已加载 {prefix}.col_ind.bin.gz ({len(col_ind)} 个元素)")
    size = max(np.max(row_ind) + 1, np.max(col_ind) + 1)
    array = scipy.sparse.csr_array((data, (row_ind, col_ind)), shape=(size, size))
    print(f"  已构造 csr_array，形状 {array.shape}，非零元数量 {array.nnz}")
    return array


# for name in [
#     "状态转移矩阵_第10抽",
#     "状态转移矩阵_前50抽但非第10抽",
#     "状态转移矩阵_第51抽及以后",
# ]:
#     prefix = f"{中间结果目录}/{name}"
#     mat = load_csr_from_bin(prefix)
#     scipy.sparse.save_npz(f"{prefix}.npz", mat)
#     print(f"  已转换并保存 {name}.npz ({mat.nnz} 个非零元)")

# print("全部矩阵就绪！")

状态转移矩阵_第10抽 = load_csr_from_bin(f"{中间结果目录}/状态转移矩阵_第10抽")
状态转移矩阵_前50抽但非第10抽 = load_csr_from_bin(f"{中间结果目录}/状态转移矩阵_前50抽但非第10抽")
状态转移矩阵_第51抽及以后 = load_csr_from_bin(f"{中间结果目录}/状态转移矩阵_第51抽及以后")

In [ ]:
中间结果目录 = "联动寻访怪猎二期中间结果"

状态转移矩阵_第10抽 = scipy.sparse.load_npz(f"{中间结果目录}/状态转移矩阵_第10抽.npz")
状态转移矩阵_前50抽但非第10抽 = scipy.sparse.load_npz(
    f"{中间结果目录}/状态转移矩阵_前50抽但非第10抽.npz"
)
状态转移矩阵_第51抽及以后 = scipy.sparse.load_npz(
    f"{中间结果目录}/状态转移矩阵_第51抽及以后.npz"
)

In [ ]:
迭代次数 = 900

初始状态 = 状态类(
    六星水位=0, 五星水位=0, UP六星水位=0, 已获取UP六星干员数量=0, 已获取UP五星干员数量=0
)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

# 由于状态数量过多，不在内存中保存历史状态分布，而改为保存 i 抽后已获取甲、乙数量的联合分布

历史获得甲乙数量联合分布 = np.zeros(
    (迭代次数 + 1, 已获取UP六星干员数量上限 + 1, 已获取UP五星干员数量上限 + 1)
)
"""`历史获得甲乙数量联合分布[n, i, j]` 表示 n 次寻访后，获取的 (甲, 乙) 干员数量恰好为 (i, j) 的概率"""
历史获得甲乙数量联合分布[0, 0, 0] = 1

状态已获取UP六星干员数量数组 = np.fromiter(
    (s.已获取UP六星干员数量 for s in 状态列表), dtype=int
)
状态已获取UP五星干员数量数组 = np.fromiter(
    (s.已获取UP五星干员数量 for s in 状态列表), dtype=int
)

for i in tqdm.notebook.tqdm(range(1, 迭代次数 + 1)):
    if i == 10:
        状态转移矩阵 = 状态转移矩阵_第10抽
    elif i <= 50:
        状态转移矩阵 = 状态转移矩阵_前50抽但非第10抽
    else:
        状态转移矩阵 = 状态转移矩阵_第51抽及以后
    当前状态分布 = 当前状态分布 @ 状态转移矩阵

    当前获得甲乙数量联合分布 = np.zeros(
        (已获取UP六星干员数量上限 + 1, 已获取UP五星干员数量上限 + 1)
    )
    np.add.at(
        当前获得甲乙数量联合分布,
        (状态已获取UP六星干员数量数组, 状态已获取UP五星干员数量数组),
        当前状态分布,
    )
    print(当前获得甲乙数量联合分布)
    历史获得甲乙数量联合分布[i] = 当前获得甲乙数量联合分布

np.savez_compressed(
    "联动寻访怪猎二期中间结果/历史获得甲乙数量联合分布.npz",
    历史获得甲乙数量联合分布=历史获得甲乙数量联合分布,
)

从 rust 输出的结果中读取数据，绘制图表

In [ ]:
# 从 Rust 计算结果中加载历史获取甲乙数量联合分布
中间结果目录 = "联动寻访怪猎二期中间结果"
迭代次数 = 900
已获取UP六星干员数量上限 = 6
已获取UP五星干员数量上限 = 6

历史获取甲乙数量联合分布 = np.load(f"{中间结果目录}/历史获取甲乙数量联合分布.npz")[
    "历史获取甲乙数量联合分布"
]

In [ ]:
dists: list[FiniteDist] = []
for n in range(已获取UP六星干员数量上限 + 1):
    cdf = np.sum(历史获取甲乙数量联合分布[:, n:, :], axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

dists[1] = FiniteDist(dists[1].pk[: 120 + 1])

draw_multi_pmf_cdf_fig(
    dists=dists[1:],
    labels=[f"{n} 名" for n in range(1, 已获取UP六星干员数量上限 + 1)],
    title="在联动寻访中获得若干名 UP 6★ 干员所需寻访次数的分布",
    x_max="auto",
    save=True,
)

draw_pmf_cdf_fig(
    dist=dists[1],
    title="在联动寻访中获得 1 名 UP 6★ 干员所需寻访次数的分布",
    x_max=None,
    save=True,
)

print_distribution_table(
    dists=dists[1:],
    labels=[f"抽{n}个" for n in range(1, 已获取UP六星干员数量上限 + 1)],
)

In [ ]:
dists: list[FiniteDist] = []
for n in range(已获取UP五星干员数量上限 + 1):
    cdf = np.sum(历史获取甲乙数量联合分布[:, :, n:], axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(
    dists=dists[1:],
    labels=[f"{n} 名" for n in range(1, 已获取UP五星干员数量上限 + 1)],
    title="在联动寻访中获得若干名 UP 5★ 干员所需寻访次数的分布",
    x_max="auto",
    save=True,
)

draw_pmf_cdf_fig(
    dist=dists[1],
    title="在联动寻访中获得 1 名 UP 5★ 干员所需寻访次数的分布",
    x_max="auto",
    save=True,
)

print_distribution_table(
    dists=dists[1:],
    labels=[f"抽{n}个" for n in range(1, 已获取UP五星干员数量上限 + 1)],
)

In [ ]:
dists = [FiniteDist([1])]

数量上限 = min(已获取UP六星干员数量上限, 已获取UP五星干员数量上限)

for n in range(1, 数量上限 + 1):
    cdf = np.sum(历史获取甲乙数量联合分布[:, n:, n:], axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(
    dists=dists[1:],
    labels=[f"各至少 {n} 次" for n in range(1, 数量上限 + 1)],
    title="在联动寻访中获得 UP 6★ 干员和 UP 5★ 干员各至少若干次所需寻访次数的分布",
    x_max="auto",
    save=True,
)

draw_pmf_cdf_fig(
    dist=dists[1],
    title="在联动寻访中获得 UP 6★ 干员和 UP 5★ 干员各至少 1 次所需寻访次数的分布",
    x_max="auto",
    save=True,
)

print_distribution_table(
    dists=dists[1:],
    labels=[f"各至少 {n} 个" for n in range(1, 数量上限 + 1)],
)